## **Notebook 2 - Feature Engineering & Model Comparison**

This notebook performs:
- Feature engineering on tweet data
- TF-IDF vectorization
- Training multiple ML models
- Cross-validation
- Model comparison using evaluation metrics

Goal: Select the best-performing baseline model.

#### **Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import nltk
import logging
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

nltk.download('punkt')


[nltk_data] Downloading package punkt to C:\Users\Aishwarya Kr
[nltk_data]     Singh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

#### **Load path for Output and models**

In [2]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)
df = pd.read_csv(OUTPUT_DIR / "train_prepared.csv")
df.head()

,id,keyword,location,text,target,text_length,word_count,text_length_chars,has_hashtag,has_mention,has_url,clean_text,tokens,original_length,clean_length
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1,69,13,69,True,False,False,our deeds are the reason of this earthquake ma...,"['our', 'deeds', 'are', 'the', 'reason', 'of',...",69,68
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1,38,7,38,False,False,False,forest fire near la ronge sask canada,"['forest', 'fire', 'near', 'la', 'ronge', 'sas...",38,37
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1,133,22,133,False,False,False,all residents asked to shelter in place are be...,"['all', 'residents', 'asked', 'to', 'shelter',...",133,130
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1,65,8,65,True,False,False,people receive wildfires evacuation orders in ...,"['people', 'receive', 'wildfires', 'evacuation...",65,56
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1,88,16,88,True,False,False,just got sent this photo from ruby alaska as s...,"['just', 'got', 'sent', 'this', 'photo', 'from...",88,85


#### **Logger Configuration**

This project uses Python logging instead of print statements.

Why?
- Debugging large datasets
- Tracking preprocessing steps
- Production ML pipelines
- Helps identify where a failure occurred

All activities will be recorded inside:
`feature_engineering.log`

In [3]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
LOG_DIR = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "feature_engineering_model_training.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8")
    ],
    force=True
)

logger = logging.getLogger(__name__)

logger.info("==== FEATURE ENGINEERING AND MODEL TRAINING PIPELINE STARTED ====")
print(f"Logging to: {LOG_FILE}")


Logging to: E:\DData\Projects\DSC\NextHikes\Python\disaster-tweet-classification-nlp-pro-7\logs\feature_engineering_model_training.log


#### **Preparing the Working Dataset**

Missing values in `keyword` and `location` are replaced with `"missing"` to simplify feature engineering. A final working dataframe is then created using the most relevant text, feature, and target columns for further analysis.


In [4]:
logger.info("Creating final working DataFrame with selected columns...")

df["tokens"] = df["clean_text"].str.split()
df["text_length_chars"] = df["text"].astype(str).apply(len)
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df["has_hashtag"] = df["text"].astype(str).str.contains(r"#", regex=True)
df["has_mention"] = df["text"].astype(str).str.contains(r"@", regex=True)
df["has_url"] = df["text"].astype(str).str.contains(r"http|www", case=False, regex=True)

work_df = (
    df.assign(
        keyword=df["keyword"].fillna("missing"),
        location=df["location"].fillna("missing")
    )[
        [
            "id", "keyword", "location", "text", "clean_text", "tokens",
            "text_length_chars", "word_count", "has_hashtag", "has_mention", "has_url", "target"
        ]
    ]
    .copy()
)

logger.info("Final working DataFrame created with shape: %s", work_df.shape)
work_df.head()

,id,keyword,location,text,clean_text,tokens,text_length_chars,word_count,has_hashtag,has_mention,has_url,target
0,1,missing,missing,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...,"[our, deeds, are, the, reason, of, this, earth...",69,13,True,False,False,1
1,4,missing,missing,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,"[forest, fire, near, la, ronge, sask, canada]",38,7,False,False,False,1
2,5,missing,missing,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...,"[all, residents, asked, to, shelter, in, place...",133,22,False,False,False,1
3,6,missing,missing,"13,000 people receive #wildfires evacuation or...",people receive wildfires evacuation orders in ...,"[people, receive, wildfires, evacuation, order...",65,8,True,False,False,1
4,7,missing,missing,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...,"[just, got, sent, this, photo, from, ruby, ala...",88,16,True,False,False,1


#### **Train-Test Split**

The dataset is divided into training and testing sets to prepare it for model development and evaluation.

- `80%` of the data is used for training
- `20%` of the data is reserved for testing
- stratified sampling is applied to preserve the target class distribution in both sets

This ensures that the training and test datasets remain balanced and suitable for reliable model performance assessment.


In [5]:
# Train/test split
logger.info("Splitting data into features and target...")
X = work_df["clean_text"]
y = work_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert split text series into proper DataFrames
train_df = pd.DataFrame({
    "text": work_df.loc[X_train.index, "text"],
    "clean_text": X_train,
    "target": y_train
})

test_df = pd.DataFrame({
    "text": work_df.loc[X_test.index, "text"],
    "clean_text": X_test,
    "target": y_test
})

logger.info("Train shape: %s", train_df.shape)
logger.info("Test shape: %s", test_df.shape)
logger.info("Train target distribution:\n%s", train_df["target"].value_counts(normalize=True))
logger.info("Test target distribution:\n%s", test_df["target"].value_counts(normalize=True))
logger.info("Data split into training and testing sets completed.")

train_df.head()

,text,clean_text,target
6234,Sassy city girl country hunk stranded in Smoky...,sassy city girl country hunk stranded in smoky...,1
326,God's Kingdom (Heavenly Gov't) will rule over ...,gods kingdom heavenly govt will rule over all ...,0
997,Mopheme and Bigstar Johnson are a problem in t...,mopheme and bigstar johnson are a problem in t...,0
7269,@VixMeldrew sounds like a whirlwind life!,sounds like a whirlwind life,0
2189,Malaysia confirms plane debris washed up on Re...,malaysia confirms plane debris washed up on re...,1


#### **TF-IDF Feature Engineering**

In [6]:
logger.info("Initializing TF-IDF vectorizer...")
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    sublinear_tf=True      # log(tf+1) instead of raw tf — reduces dominance of
                           # high-frequency keywords like 'weather', 'fire', 'flood'
                           # that appear in BOTH disaster and non-disaster tweets
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
logger.info("TF-IDF vectorization completed.")
logger.info("TF-IDF feature matrix shapes - Train: %s, Test: %s", X_train_tfidf.shape, X_test_tfidf.shape)


#### **Save Prepared Datasets**

The processed datasets are saved for later use in model training and evaluation.

- the training dataset is stored separately
- the testing dataset is stored separately
- the complete prepared dataset is also saved as a reference copy

This makes the workflow more organized and helps reuse the processed data in future steps.


In [7]:
train_file = OUTPUT_DIR / "train_dataset_prepared.csv"
test_file = OUTPUT_DIR / "test_dataset_prepared.csv"
full_file = OUTPUT_DIR / "complete_dataset_prepared.csv"

train_df.to_csv(train_file, index=False)
test_df.to_csv(test_file, index=False)
work_df.to_csv(full_file, index=False)

print("Saved:")
print("-", train_file)
print("-", test_file)
print("-", full_file)
logger.info("Saved prepared datasets to output directory.")
train_df.info()
test_df.info()
work_df.info()

Saved:
- ..\data\train_dataset_prepared.csv
- ..\data\test_dataset_prepared.csv
- ..\data\complete_dataset_prepared.csv
<class 'pandas.core.frame.DataFrame'>
Index: 6090 entries, 6234 to 7569
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        6090 non-null   object
 1   clean_text  6090 non-null   object
 2   target      6090 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 190.3+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 1523 entries, 4863 to 2344
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        1523 non-null   object
 1   clean_text  1523 non-null   object
 2   target      1523 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 47.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------    

#### **Model Initialization**

This step defines the machine learning models that will be trained and compared for disaster tweet classification.

The selected models represent different learning approaches:
- `LinearSVC` for linear margin-based classification
- `LogisticRegression` for probabilistic linear classification
- `MultinomialNB` for frequency-based text classification
- `RandomForestClassifier` for ensemble-based decision learning

Using multiple models makes it possible to compare their performance and identify the most suitable approach for the task.


In [8]:
logger.info("Initializing models...")

# LinearSVC does not produce probabilities natively.
# CalibratedClassifierCV wraps it with isotonic regression so it
# outputs real predict_proba values — fixing the 'always Disaster'
# bug caused by converting raw decision_function margins via sigmoid.
#
# class_weight='balanced' accounts for the slight class imbalance
# in the dataset (more non-disaster tweets than disaster ones).

models = {
    "linear_svc":          CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=2000)),
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "multinomial_nb":      MultinomialNB(),
    "random_forest":       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
}
logger.info("Models initialized with calibration and class_weight='balanced'.")


#### **Model Training and Evaluation**

This step trains each selected model on the TF-IDF-transformed training data and evaluates its performance on the test set.

For each model, the following metrics are calculated:
- `accuracy` to measure overall correctness
- `precision` to evaluate prediction quality for the positive class
- `recall` to measure how well disaster tweets are identified
- `f1-score` to balance precision and recall
- `cross-validation F1` to assess performance consistency on the training data

The results are stored in a summary list for later comparison across models.


In [9]:
logger.info("Model training and evaluation started...")
results = []

for name, model in models.items():
    
    # Train
    model.fit(X_train_tfidf, y_train)
    
    # Predict
    y_pred = model.predict(X_test_tfidf)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    logger.info("Model: %s - Test Accuracy: %.4f, Precision: %.4f, Recall: %.4f, F1 Score: %.4f", name, accuracy, precision, recall, f1)
    
    # Cross-validation F1
    cv_f1 = cross_val_score(model, X_train_tfidf, y_train, cv=3, scoring="f1").mean()
    
    results.append({
        "model": name,
        "cv_f1": round(cv_f1, 4),
        "test_accuracy": round(accuracy, 4),
        "test_precision": round(precision, 4),
        "test_recall": round(recall, 4),
        "test_f1": round(f1, 4)
    })
logger.info("Model training and evaluation completed.")

#### **Model Comparison**

This step organizes the evaluation results of all trained models into a single dataframe for comparison.

- the results are converted into tabular form
- the models are sorted by `test_f1` in descending order
- the best-performing model appears at the top of the table

This makes it easier to compare performance across models and identify the most effective classifier for disaster tweet prediction.


In [10]:
logger.info("Model comparison started...")
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="test_f1", ascending=False)
logger.info("Model comparison completed.")
results_df

,model,cv_f1,test_accuracy,test_precision,test_recall,test_f1
1,logistic_regression,0.7430,0.7951,0.7575,0.7691,0.7633
0,linear_svc,0.7208,0.7965,0.7857,0.7232,0.7532
2,multinomial_nb,0.7164,0.8116,0.8692,0.6606,0.7507
3,random_forest,0.6992,0.7978,0.8204,0.6774,0.7420


#### **Best Model Selection**

This step identifies the best-performing model based on the highest test `F1-score`.

- the top model name is selected from the sorted comparison table
- the corresponding trained model is retrieved from the model dictionary
- the selected model is stored for further analysis and prediction

This ensures that the final workflow proceeds with the most effective classifier from the evaluated models.


In [11]:
logger.info("Best model based on test F1 score: %s", results_df.iloc[0]["model"])
best_model_name = results_df.iloc[0]["model"]
best_model_name
best_model = models[best_model_name]
best_f1 = results_df.iloc[0]["test_f1"]
best_cv = results_df.iloc[0]["cv_f1"]
logger.info("Best model selected: %s", best_model_name)

#### **Save Vectorizer and Best Model**

This step saves the fitted TF-IDF vectorizer and the selected best-performing model to disk.

- the TF-IDF vectorizer is stored for future text transformation
- the best model is saved for later prediction and deployment
- the saved files can be loaded again without retraining

This helps preserve the trained pipeline and supports reuse in inference or application development.


In [12]:
logger.info("Saving TF-IDF vectorizer and best model to disk...")
pickle.dump(tfidf, open("../models/tfidf.pkl", "wb"))
pickle.dump(best_model, open("../models/best_model.pkl", "wb"))
logger.info("Best model saved: %s", best_model_name)

#### **Save Model Comparison Results**

This step saves the model evaluation summary as a CSV file for future reference.

The exported file contains the performance metrics of all tested models, making it easier to review, compare, and document the results outside the notebook.


In [13]:
results_df.to_csv("../data/model_comparison_results.csv", index=False)
logger.info("Model comparison results saved to: %s", "../data/model_comparison_results.csv")
logger.info("==== FEATURE ENGINEERING AND MODEL TRAINING PIPELINE COMPLETED ====")
results_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 1 to 3
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   model           4 non-null      object 
 1   cv_f1           4 non-null      float64
 2   test_accuracy   4 non-null      float64
 3   test_precision  4 non-null      float64
 4   test_recall     4 non-null      float64
 5   test_f1         4 non-null      float64
dtypes: float64(5), object(1)
memory usage: 224.0+ bytes


---

#### **Model Performance Analysis**

The performance of different machine learning models is summarized in the table above.

#### Key Observations:

- Multiple models were evaluated using cross-validation and test metrics.
- Performance varies across models depending on their ability to handle sparse TF-IDF features.
- Linear models generally perform well for text classification tasks.

#### Best Model Selection:

Based on the evaluation results:

In [14]:
print(f"""
Best Model Selection:

Best Model: {best_model_name}
Test F1 Score: {best_f1}
CV F1 Score: {best_cv}
""")


Best Model Selection:

Best Model: logistic_regression
Test F1 Score: 0.7633
CV F1 Score: 0.743



This model demonstrates the best balance between precision and recall, making it the most suitable for disaster tweet classification.

### Insights:

- Models with higher F1-score are preferred as they balance both false positives and false negatives.
- Cross-validation ensures the model is stable and generalizes well to unseen data.

---

The selected model will be further optimized in the next notebook.

---
#### **Notebook 2 Summary**

In this notebook, we successfully:

#### Feature Engineering:
- Extracted structural features such as:
  - Text length
  - Word count
  - Presence of hashtags, mentions, and URLs

#### Text Representation:
- Applied **TF-IDF vectorization** with n-grams to capture contextual information from tweets.

#### Model Training:
- Trained multiple machine learning models:
  - Linear SVC
  - Logistic Regression
  - Multinomial Naive Bayes
  - Random Forest

#### Model Evaluation:
- Evaluated models using:
  - Accuracy
  - Precision
  - Recall
  - F1-score
  - Cross-validation (cv_f1)

#### Model Selection:
- Selected the best-performing model based on **F1-score and generalization performance**.

---

#### Conclusion

Linear models demonstrated superior performance for text classification tasks using TF-IDF features. The selected model provides a strong baseline for disaster tweet classification.

---

#### Next Steps

In the next notebook (**03_ML_Model_TFIDF.ipynb**), we will:

- Perform **hyperparameter tuning** (GridSearchCV)
- Improve model performance
- Reduce overfitting
- Finalize the best ML model for deployment

---

#### Project Relevance

This step directly supports the project objective of building an accurate and robust classification model for identifying disaster-related tweets, as required in the problem statement.